# Visualise the eigenvalues of the hybridisation matrix
For a 3×2 pairs system, diagonalises the hybridisation matrix at every frequency between 260 and 310 kHz and plots its eigenvalues (magnitudes against frequency, and in the complex plane coloured by frequency), the components of its eigenvectors, and finally the homodyne and heterodyne spectra (Numerical H) over the same range.

Runtime: about 1 min (`FINE_GRID = True`, the original 0.001 kHz grid: about ten times longer).

# START

# Visualise eigenvalues of the hybridisation matrix

In [ ]:
import scipy
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
cavity_kwargs = dict(Δ=-450*kHz, L=0.96*cm, Finesse=40_000, θhom=π/4)
toy_params = dict(Ω_model='pairs', toy_gx=-40*kHz, toy_gy=-40*kHz)

# TODO: could isolate each peak to see behaviour of the eigenvector whose eigenvalue contributes the most?
# (this usually seems to be λ1)

# FINE_GRID = True is the original 0.001 kHz grid (50,000 frequencies, several minutes);
# False uses 0.01 kHz (5,000 frequencies), enough to resolve every resonance here
FINE_GRID = False

# ω_range = np.arange(0, 400, 0.01) * kHz
ω_range = np.arange(260, 310, 0.001 if FINE_GRID else 0.01) * kHz

# 3x2 takes about 2 mins (FINE_GRID)
system = '3×2'
Nd = system.split('×')
N, d = int(Nd[0]), int(Nd[1])
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    ω_range=ω_range,
    gas_kwargs=gas_kwargs,
    cavity_kwargs=cavity_kwargs,
    toy_params=toy_params,
)

methods = [QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION]
spectra = [
    SpectrumType.HOMODYNE,
    SpectrumType.HETERODYNE,
]

# debug on resonance
if 0:
    H = mtrap.hybridisation_matrix(methods[0], 300*kHz)
    λi, Q = scipy.linalg.eig(H, left=False, right=True)
    λi2 = scipy.linalg.eigvals(H)

    print(f'{H=}')
    print()
    print(f'{λi=}')
    print(f'{λi2=}')
    print()
    print(f'{Q.shape=}')
    print(f'{Q=}')
    print(f'{Q[:, 0]=}')
    print(f'{Q[:, 1]=}')
    print()
    print(f'{Q[:, 0][0]=}')
    print(f'{Q[:, 1][0]=}')
    print()
    print(f'{Q[:, 0][1]=}')
    print(f'{Q[:, 1][1]=}')
    print()
    print(f'{H @ Q[:, 0]=}')
    print(f'{λi[0] * Q[:, 0]=}')
    print()
    print(f'{H @ Q[:, 1]=}')
    print(f'{λi[1] * Q[:, 1]=}')

# [0] gives the list of eigenvalues
# [1] gives the array where each column [1][:, ii] is eigenvector #ii + 1
eigensystem_across_ω_range = [scipy.linalg.eig(mtrap.hybridisation_matrix(methods[0], ω)) for ω in ω_range]

####################################################################################################
# EIGENVALUE PLOTS
####################################################################################################
plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5), dpi=150)
plt.title('Spectrum of mean of all eigenvalue magnitudes')
plt.xlabel('Frequency [kHz]')
plt.xlim(ω_range.min() / kHz, ω_range.max() / kHz)
plt.scatter(
    ω_range / kHz,
    [np.sum(np.abs(eigensystem_across_ω_range[jj][0])) / mtrap.num_dimensions for jj in range(len(ω_range))],
    label=rf'mean$(|\lambda_i|)$',
    marker='.', alpha=0.5, linewidths=0,
    c=ω_range, cmap='gist_rainbow',
)
plt.yscale('log')
plt.grid(alpha=0.1)
plt.legend()
plt.show()


plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5), dpi=150)
plt.title('All eigenvalues $\lambda_i$ in complex plane')
plt.xlabel(f'Re[$\lambda_i$]')
plt.ylabel(f'Im[$\lambda_i$]')
plt.axhline(0, color=black, alpha=0.15)
plt.axvline(0, color=black, alpha=0.15)
for ii in range(mtrap.num_dimensions):
    plt.scatter(
        [np.real(eigensystem_across_ω_range[ωω][0][ii]) for ωω in range(len(ω_range))],
        [np.imag(eigensystem_across_ω_range[ωω][0][ii]) for ωω in range(len(ω_range))],
        label=rf'$\lambda_{{{ii + 1}}}$',
        marker='.', alpha=0.5, linewidths=0,
        c=ω_range, cmap='gist_rainbow',
    )
plt.gca().set_aspect('equal', 'box')
plt.grid(alpha=0.1)
plt.legend(ncol=2)
plt.show()


plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5), dpi=150)
plt.title('All eigenvalues $\lambda_i$ in complex plane (zoomed)')
plt.xlim(-10, 10)
plt.ylim(-10, 10)
plt.xlabel(f'Re[$\lambda_i$]')
plt.ylabel(f'Im[$\lambda_i$]')
plt.axhline(0, color=black, alpha=0.15)
plt.axvline(0, color=black, alpha=0.15)
for ii in range(mtrap.num_dimensions):
    plt.scatter(
        [np.real(eigensystem_across_ω_range[ωω][0][ii]) for ωω in range(len(ω_range))],
        [np.imag(eigensystem_across_ω_range[ωω][0][ii]) for ωω in range(len(ω_range))],
        label=rf'$\lambda_{{{ii + 1}}}$',
        marker='.', alpha=0.5, linewidths=0,
        c=ω_range, cmap='gist_rainbow',
    )
plt.gca().set_aspect('equal', 'box')
plt.grid(alpha=0.1)
plt.legend(ncol=2)
plt.show()


plt.figure(figsize=(LATEX_FIG_WIDTH, 3.5), dpi=150)
plt.title('Spectrum of each eigenvalue`s magnitude')
plt.xlabel('Frequency [kHz]')
plt.xlim(ω_range.min() / kHz, ω_range.max() / kHz)
plt.plot(
    ω_range / kHz,
    [np.abs(eigensystem_across_ω_range[ωω][0]) for ωω in range(len(ω_range))],
    label=[rf'$|\lambda_{{{ii + 1}}}|$' for ii in range(mtrap.num_dimensions)],
    marker='.', alpha=0.1, markersize=0.5, lw=0,
)
plt.yscale('log')
plt.grid(alpha=0.1)
plt.legend()
plt.show()

####################################################################################################
# EIGENVECTOR PLOTS
####################################################################################################
fig = plt.figure(layout='constrained', figsize=(15, 3 * N), dpi=150)
ax_dict = fig.subplot_mosaic(
    [['all'] * N + [f'q{2 * trap_index + 1}'] + [f'q{2 * trap_index + 2}'] for trap_index in range(mtrap.num_traps)]
)
all_ax = ax_dict['all']
all_ax.set_title('All eigenvectors $q_i$')
all_ax.set_xlabel(f'Re[$q_i$]')
all_ax.set_ylabel(f'Im[$q_i$]')
all_ax.axhline(0, color=black, alpha=0.15)
all_ax.axvline(0, color=black, alpha=0.15)

for ii in range(mtrap.num_dimensions):
    ax = ax_dict[f'q{ii + 1}']
    ax.grid(alpha=0.1)

    for kk in range(mtrap.num_dimensions):
        ax.scatter(
            [np.real(eigensystem_across_ω_range[ωω][1][:, ii][kk]) for ωω in range(len(ω_range))],
            [np.imag(eigensystem_across_ω_range[ωω][1][:, ii][kk]) for ωω in range(len(ω_range))],
            label=f'$q_{ii + 1}$[${mtrap.dimensions[mtrap.labels[kk]].latex}$]',
            marker='.', alpha=0.5, linewidths=0,
        )
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_title(f'$q_{ii + 1}$')
        ax.set_xlabel(f'Re[$q_{ii + 1}$]')
        ax.set_ylabel(f'Im[$q_{ii + 1}$]')
        ax.legend(loc='lower right', ncol=4, prop=dict(size=7))

        all_ax.scatter(
            [np.real(eigensystem_across_ω_range[ωω][1][:, ii][kk]) for ωω in range(len(ω_range))],
            [np.imag(eigensystem_across_ω_range[ωω][1][:, ii][kk]) for ωω in range(len(ω_range))],
            label=f'$q_{ii + 1}$[${mtrap.dimensions[mtrap.labels[kk]].latex}$]',
            marker='.', alpha=0.5, linewidths=0,
        )

all_ax.grid(alpha=0.1)
all_ax.legend(loc='lower right', ncol=4, prop=dict(size=7))
plt.show()

fig = plt.figure(layout='constrained', figsize=(2 * LATEX_FIG_WIDTH, 5 * N), dpi=150)
ax_dict = fig.subplot_mosaic(
    [
        [f'Re[x,y for trap {trap_index + 1}]', f'Im[x,y for trap {trap_index + 1}]']
        for trap_index in range(mtrap.num_traps)
    ],
)

# plot each trap's x, y coordinates (real and imaginary) that are contained in each eigenvector
for trap_index in range(mtrap.num_traps):
    Re_ax = ax_dict[f'Re[x,y for trap {trap_index + 1}]']
    Im_ax = ax_dict[f'Im[x,y for trap {trap_index + 1}]']
    xx = mtrap.Cartesian_axes_per_trap * trap_index
    yy = xx + 1
    for ii in range(mtrap.num_dimensions):
        Re_ax.scatter(
            [np.real(eigensystem_across_ω_range[ωω][1][:, ii][xx]) for ωω in range(len(ω_range))],
            [np.real(eigensystem_across_ω_range[ωω][1][:, ii][yy]) for ωω in range(len(ω_range))],
            label=f'Re[x{trap_index + 1},y{trap_index + 1}] from eigenvector #{ii + 1}',
            marker='.', alpha=0.5, linewidths=0,
            # alpha=[0.999 ** jj for jj in range(len(ω_range))]
            # alpha=[abs(ω) / (1.1 * max_ω) for ω in ω_range],
        )
        Im_ax.scatter(
            [np.imag(eigensystem_across_ω_range[ωω][1][:, ii][xx]) for ωω in range(len(ω_range))],
            [np.imag(eigensystem_across_ω_range[ωω][1][:, ii][yy]) for ωω in range(len(ω_range))],
            # label=f'Im[x{trap_index + 1},y{trap_index + 1}] from eigenvector #{ii + 1}',
            marker='.', alpha=0.5, linewidths=0,
            # alpha=[0.999 ** jj for jj in range(len(ω_range))]
            # alpha=[abs(ω) / (1.1 * max_ω) for ω in ω_range],
        )

    for ax in [Re_ax, Im_ax]:
        tt = 'Re' if ax == Re_ax else 'Im'
        ax.set_title(rf'{tt}[$x_{{{trap_index + 1}}}$, $y_{{{trap_index + 1}}}$] of all $q_i$')
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_xticks([-1, -0.5, 0, 0.5, 1])
        ax.set_yticks([-1, -0.5, 0.5, 1])
        ax.grid(alpha=0.1)
        ax.axhline(0, color=black, alpha=0.15)
        ax.axvline(0, color=black, alpha=0.15)
        # ax.legend()

plt.show()

####################################################################################################
# SPECTRA (plotted directly; the old image-diff helper needed OpenCV and only one method is used here)
####################################################################################################
fig, axes = plt.subplots(len(spectra), 1, figsize=(LATEX_FIG_WIDTH, 3 * len(spectra)), dpi=150, sharex=True, gridspec_kw=dict(hspace=0))
for ax, spectrum_type in zip(axes, spectra):
    match spectrum_type:
        case SpectrumType.HOMODYNE:
            vals = mtrap.homodyne_spectrum(methods[0])
        case SpectrumType.HETERODYNE:
            vals = mtrap.heterodyne_spectrum(methods[0])
    ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], 1e-1, 1e4, color=black, alpha=0.1, ls='--')
    ax.semilogy(ω_range / kHz, vals, color=blue, label=f'{repr(spectrum_type)} via {repr(methods[0])}')
    ax.set_xlim(ω_range.min() / kHz, ω_range.max() / kHz)
    ax.set_ylabel('[photons/Hz]')
    ax.legend(loc='upper left', prop=dict(size=8))
axes[-1].set_xlabel('Frequency [kHz]')
plt.show()


# END